In [3]:
import numpy as np
import pandas as pd

df_master = pd.read_csv("../data/df_master_cleaned.csv")

# Lifetime Value (LTV)
df_master['LTV'] = df_master['MonthlyCharges'] * df_master['tenure']

# Annualized Run Rate (ARR) per customer
df_master['ARR'] = df_master['MonthlyCharges'] * 12

# Quantitative justification for risk multiplier

# 1. Overall Portfolio Baseline Churn Rate
baseline_churn = (df_master['Churn'] == 'Yes').mean()
print(f"Overall Baseline Churn Rate: {baseline_churn:.2%}")

# 2. Segment Analysis: Month-to-month + Fiber optic Internet
vulnerable_segment = df_master[(df_master['Contract'] == 'Month-to-month') & (df_master['InternetService'] == 'Fiber optic')]
vulnerable_churn = (vulnerable_segment['Churn'] == 'Yes').mean()
print(f"Vulnerable Segment Churn Rate: {vulnerable_churn:.2%}")

# 3. Empirically Derived Multiplier Calculation
ratio = vulnerable_churn / baseline_churn
print(f"Calculated Risk Ratio: {ratio:.2f}x (Normalized conservatively to 1.5x)")

# Applying Dynamic Financial Risk Multiplier
df_master['Risk_Weight_Multiplier'] = np.where(
    (df_master['Contract'] == 'Month-to-month') & (df_master['InternetService'] == 'Fiber optic'),
    1.5,
    1.0
)

# At-Risk Adjusted Monthly Revenue
df_master['At_Risk_Weighted_Revenue'] = df_master['MonthlyCharges'] * df_master['Risk_Weight_Multiplier']

# Display summary of engineered financial metrics
print("\n--- Financial Metrics Engineering Complete ---")
print(f"Total Portfolio ARR: ${df_master['ARR'].sum():,.2f}")
print(f"Average Customer LTV: ${df_master['LTV'].mean():,.2f}")
print(f"Total High-Risk Weight Accounts (1.5x): {(df_master['Risk_Weight_Multiplier'] == 1.5).sum()}")

# Inspect financial summary grouped by Churn status
financial_summary = df_master.groupby('Churn').agg(
    Account_Count=('customerID', 'count'),
    Total_Monthly_Revenue=('MonthlyCharges', 'sum'),
    Total_ARR=('ARR', 'sum'),
    Average_LTV=('LTV', 'mean'),
    Total_Historical_Revenue=('TotalCharges', 'sum')
).reset_index()

financial_summary

df_master.to_csv("../data/df_master_financials.csv", index=False)

Overall Baseline Churn Rate: 26.54%
Vulnerable Segment Churn Rate: 54.61%
Calculated Risk Ratio: 2.06x (Normalized conservatively to 1.5x)

--- Financial Metrics Engineering Complete ---
Total Portfolio ARR: $5,473,399.20
Average Customer LTV: $2,279.58
Total High-Risk Weight Accounts (1.5x): 2128
